To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

**Read our [Gemma 3 blog](https://unsloth.ai/blog/gemma3) for what's new in Unsloth and our [Reasoning blog](https://unsloth.ai/blog/r1-reasoning) on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


In [37]:
upper_vowel = ["่", "้", "๊", "๋", "็", "ิ", "ี", "ึ", "ื", "ุ", "ู", "์", "ั"]
for vowel in upper_vowel:
    print(vowel)

่
้
๊
๋
็
ิ
ี
ึ
ื
ุ
ู
์
ั


In [ ]:
def print_poem_formatted(text):
    # Find the maximum width of the lines
    lines = text.split('\n')
    
    max_width = -1
    for line in lines:
        tabs = line.split('\t')
        for tab in tabs:
            tab_length = len(tab) - sum(1 for char in tab if char in upper_vowel)
            max_width = max(max_width, tab_length)
    
    for line in lines:
        tabs = line.split('\t')
        for tab in tabs:
            vowels_count = sum(1 for char in tab if char in upper_vowel)
            char_length = len(tab) - vowels_count
            
            print(tab, end='')
            for _ in range(char_length, max_width):
                print(' ', end='')
        print()

In [69]:
print_poem_formatted(text_a)

<klon8> สั่งกำชับสรรพเสร็จเสนา<r>[O]หน้า</r>       ช่วยดล<r>[O]จิต</r>เย็นใจเยือกประทาน<r>[a]โปรด</r>
อย่าช่วยเก็บต้อนผู้คนเหมือนเช่น<r>[a]เคย</r>          คิดตรึก<r>[a]ตรา</r>ตั้งแต่ช่าง<r>[a]ช้า</r>         
ยิ่งลนลานต้องรีบเดินเดิน<r>[a]ข้าง</r>               อย่าด่วน<r>[a]ด่า</r>ปลุกทุกคนให้<r>[a][j]ได้</r>    
ไปด้วยเป็นนายให้ถึงจวน                            


In [70]:
print_poem_formatted(text_b)

<klon8> สั่งกำชับสรรพเสร็จเสด็จ<r>[a][t]คลาด</r>         คัม<r>[a][t]ภีร์</r>เยื้องย่องเข้าห้อง<r>[a][j]หาย</r>      
กระซิบสั่งพี่เลี้ยงพระเถ้า<r>[a][j]ชาย</r>                  เจ้าจง<r>[a][j]กลาย</r>เกลียวกล่อมให้พร้อม<r>[a][j]ใจ</r>
แล้วเลิกม่านบ้านพลับพลา<r>[a][N]ตั้ง</r>                   เสียงส<r>[a][N]นั่ง</r>น้อยหรือพูดเสน่<r>[a][j]ได้</r>      
ครั้น                                                 
                                                    


### Installation

In [31]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [2]:
!pip install -q wandb
!pip install --upgrade typing-extensions
!pip install transformers==4.50.3


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
WANDB_API_KEY = "paste_your_wandb_api_key_here"
HUGGINGFACE_API_KEY = "paste_your_huggingface_api_key_here"

In [2]:
experiment_name = "llama3.2-typhoon2-1b-full-training-phonetic"

In [3]:
import wandb

wandb.login(key=WANDB_API_KEY)
wandb.init(project="nlp-phonetic", name=experiment_name, resume="allow")

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pongsaky to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


### Get new phonetic token, training and valida data

In [3]:
import os

os.environ["UNSLOTH_IS_PRESENT"] = "1"

In [4]:
from unsloth_zoo import loss_utils

In [5]:
import json

def get_data_from_json(file_path):
  with open(file_path, "r", encoding="utf-8") as f:
      data = json.load(f)
  return data

phonetic_token = get_data_from_json("./data/phonetic_token.json")
training_data = get_data_from_json("./data/training_data.json")
validation_data = get_data_from_json("./data/valid_data.json")
test_data_input = get_data_from_json("./data/test_inputs.json")

In [7]:
len(training_data)

9970

### Unsloth

In [8]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "scb10x/llama3.2-typhoon2-1b",
    # model_name = "./checkpoints-llama3.2-typhoon2-1b-lora-unfreeze-embedding-phonetic-runpod/checkpoint-234", # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = max_seq_length,
    dtype = torch.float16,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=True,
    token = HUGGINGFACE_API_KEY, # use one if using gated models like meta-llama/Llama-2-7b-hf,
)

Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.673 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Float16 full finetuning uses more memory since we upcast weights to float32.


In [9]:
model.vocab_size, tokenizer.vocab_size

(128256, 128000)

In [ ]:
from unsloth import add_new_tokens

add_new_tokens(model, tokenizer, phonetic_token)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [11]:
model.vocab_size, tokenizer.vocab_size

(128320, 128000)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj", "lm_head", "embed_tokens"],
    lora_alpha = 8,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


Unsloth 2025.3.19 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


In [12]:
for name, param in model.named_parameters():
    print(name, param.requires_grad)

model.embed_tokens.weight True
model.layers.0.self_attn.q_proj.weight True
model.layers.0.self_attn.k_proj.weight True
model.layers.0.self_attn.v_proj.weight True
model.layers.0.self_attn.o_proj.weight True
model.layers.0.mlp.gate_proj.weight True
model.layers.0.mlp.up_proj.weight True
model.layers.0.mlp.down_proj.weight True
model.layers.0.input_layernorm.weight True
model.layers.0.post_attention_layernorm.weight True
model.layers.1.self_attn.q_proj.weight True
model.layers.1.self_attn.k_proj.weight True
model.layers.1.self_attn.v_proj.weight True
model.layers.1.self_attn.o_proj.weight True
model.layers.1.mlp.gate_proj.weight True
model.layers.1.mlp.up_proj.weight True
model.layers.1.mlp.down_proj.weight True
model.layers.1.input_layernorm.weight True
model.layers.1.post_attention_layernorm.weight True
model.layers.2.self_attn.q_proj.weight True
model.layers.2.self_attn.k_proj.weight True
model.layers.2.self_attn.v_proj.weight True
model.layers.2.self_attn.o_proj.weight True
model.lay

<a name="Data"></a>
### Data Prep
We now use the `Llama-3.1` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. But we convert it to HuggingFace's normal multiturn format `("role", "content")` instead of `("from", "value")`/ Llama-3 renders multi turn conversations like below:

```
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Hello!<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Hey there! How are you?<|eot_id|><|start_header_id|>user<|end_header_id|>

I'm great thanks!<|eot_id|>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3` and more.

In [12]:
tokenizer.pad_token

'<|end_of_text|>'

In [13]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": training_data})
valid_dataset = Dataset.from_dict({"text": validation_data})

train_dataset = train_dataset.map(lambda x :  { "text" : x["text"] }).shuffle(seed=42)
valid_dataset = valid_dataset.map(lambda x :  { "text" : x["text"] }).shuffle(seed=42)

train_dataset, valid_dataset

Map:   0%|          | 0/9970 [00:00<?, ? examples/s]

Map:   0%|          | 0/1035 [00:00<?, ? examples/s]

(Dataset({
     features: ['text'],
     num_rows: 9970
 }),
 Dataset({
     features: ['text'],
     num_rows: 1035
 }))

In [14]:
train_dataset["text"][0]

'<klon8> ครานั้นท่านยายบุษ<r>[a]บา</r>\tปลอบลูกสาว<r>[a]ว่า</r>อย่าสะ<r>[U][n]อื้น</r>\nพ่อแม่ก็ได้สั่งไว้ยั่ง<r>[U][n]ยืน</r>\tหม่อม<r>[U][n]หมื่น</r>เธอก็รับปฏิ<r>[a][n]ญาณ</r>\nแต่ใจแม่นี้ยังกริ่งอยู่สิ่ง<r>[U][N]หนึ่ง</r>\tกลัวจะ<r>[U][N]หึง</r>กันวุ่นวายอายชาว<r>[a][n]บ้าน</r>\nอันเมียสองต้องห้ามตามโบ<r>[a][n]ราณ</r>\tเป็นกับใครก็รำคาญไม่เว้น<r>[o][n]คน</r>\nแม่สอนเจ้ามาแต่น้อยกว่าร้อย<r>[a][n]พัน</r>\tสุดสำ<r>[a][n]คัญ</r>แต่เพียงอดนั้นเป็น<r>[o][n]ต้น</r>\nอย่าทำชั่วเพราะว่าตัวของตัว<r>[o][n]จน</r>\tเขาเปรียบเทียบจงสู้ทนต้องเกรง<r>[ua]กลัว</r>\nใครจะด่าเจาะจังก็ชั่ง<r>[a][w]เขา</r>\tจงอด<r>[a][w]เอา</r>อย่าสำออยคอยฟ้อง<r>[ua]ผัว</r>\nอันคนดีนานดอกจึงออก<r>[ua]ตัว</r>\tถ้าคน<r>[ua]ชั่ว</r>เขาคงเห็นเป็นไป<r>[e][N]เอง</r>\n'

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [15]:
from transformers import TrainerCallback
import torch

class TextOutputCallback(TrainerCallback):
    """
    A custom callback that generates and prints model outputs every `every_n_steps` steps.
    """
    def __init__(
        self,
        model,
        tokenizer,
        test_data_input: list[str],
        every_n_steps: int = 10,
        max_new_tokens: int = 468,
        temperature: float = 1,
        top_p: float = 0.95,
        top_k: int = 64
    ):
        """
        Args:
            model: The model to use for generation
            tokenizer: The tokenizer to use
            test_data_input (List[str]): List of input text for generation
            every_n_steps (int): Interval between generations
            max_new_tokens (int): Maximum tokens to generate
            temperature (float): Sampling temperature
            top_p (float): Nucleus sampling parameter
            top_k (int): Top-k sampling parameter
        """
        self.model = model
        self.tokenizer = tokenizer
        self.test_data_input = test_data_input
        self.every_n_steps = every_n_steps
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p
        self.top_k = top_k

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every_n_steps == 0:
            print(f"\n=== Generating output at step {state.global_step} ===")

            rand_idx = int(torch.rand(1) * len(self.test_data_input))
            sample_text = self.test_data_input[rand_idx]

            # Prepare inputs
            inputs = self.tokenizer(
                [sample_text],
                return_tensors="pt",
                padding=True,
                truncation=True
            ).to(self.model.device)

            # Generate outputs
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=self.max_new_tokens,
                    temperature=self.temperature,
                    top_p=self.top_p,
                    top_k=self.top_k,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            # Decode and print
            generated_text = self.tokenizer.decode(
                outputs[0],
                skip_special_tokens=False,
                clean_up_tokenization_spaces=True
            )

            print(f"Input: {sample_text}")
            print(f"Output: {generated_text}")
            print("=" * 80)

        return control

In [16]:
import numpy as np

In [17]:
epoch = 3
batch_size = 24
accumulation_steps = 3
number_time_text_output = 20
number_of_eval = 20
number_of_save_step_per_epoch = 1

total_steps = int(len(train_dataset) / (batch_size * accumulation_steps)) * epoch
warmup_steps = int(total_steps * 0.1)
every_n_steps = int(total_steps / number_time_text_output)
eval_steps = int(total_steps / number_of_eval)
save_steps = int(np.ceil((total_steps / ( number_of_save_step_per_epoch * epoch)) / eval_steps) * eval_steps)

In [18]:
total_steps, warmup_steps, every_n_steps, eval_steps, save_steps, experiment_name

(414, 41, 20, 20, 140, 'llama3.2-typhoon2-1b-full-training-phonetic')

In [19]:
from transformers import EarlyStoppingCallback

callbacks = [
    EarlyStoppingCallback(early_stopping_patience=5),
    TextOutputCallback(
        model=model,
        tokenizer=tokenizer,
        every_n_steps=every_n_steps,
        test_data_input=test_data_input
    )
]

In [20]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset= valid_dataset,
    dataset_text_field = "text",
    metric_for_best_model="eval_loss",
    eval_strategy='steps',
    save_strategy='steps',
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    callbacks = callbacks,
    args = TrainingArguments(
        per_device_train_batch_size = batch_size,
        gradient_accumulation_steps = accumulation_steps,
        warmup_steps = warmup_steps,
        num_train_epochs = epoch, # Set this for 1 full training run.,
        learning_rate = 2e-4,
        # fp16 = not is_bfloat16_supported(),
        # bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir=f"./checkpoints-{experiment_name}",
        report_to = "wandb", # Use this for WandB etc,
        save_steps=save_steps,
        eval_steps=eval_steps,
        run_name=experiment_name,
        eval_strategy="steps",
        load_best_model_at_end=True,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/9970 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1035 [00:00<?, ? examples/s]

We verify masking is actually done:

In [21]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

'<|begin_of_text|><klon8> ครานั้นท่านยายบุษ<r>[a]บา</r>\tปลอบลูกสาว<r>[a]ว่า</r>อย่าสะ<r>[U][n]อื้น</r>\nพ่อแม่ก็ได้สั่งไว้ยั่ง<r>[U][n]ยืน</r>\tหม่อม<r>[U][n]หมื่น</r>เธอก็รับปฏิ<r>[a][n]ญาณ</r>\nแต่ใจแม่นี้ยังกริ่งอยู่สิ่ง<r>[U][N]หนึ่ง</r>\tกลัวจะ<r>[U][N]หึง</r>กันวุ่นวายอายชาว<r>[a][n]บ้าน</r>\nอันเมียสองต้องห้ามตามโบ<r>[a][n]ราณ</r>\tเป็นกับใครก็รำคาญไม่เว้น<r>[o][n]คน</r>\nแม่สอนเจ้ามาแต่น้อยกว่าร้อย<r>[a][n]พัน</r>\tสุดสำ<r>[a][n]คัญ</r>แต่เพียงอดนั้นเป็น<r>[o][n]ต้น</r>\nอย่าทำชั่วเพราะว่าตัวของตัว<r>[o][n]จน</r>\tเขาเปรียบเทียบจงสู้ทนต้องเกรง<r>[ua]กลัว</r>\nใครจะด่าเจาะจังก็ชั่ง<r>[a][w]เขา</r>\tจงอด<r>[a][w]เอา</r>อย่าสำออยคอยฟ้อง<r>[ua]ผัว</r>\nอันคนดีนานดอกจึงออก<r>[ua]ตัว</r>\tถ้าคน<r>[ua]ชั่ว</r>เขาคงเห็นเป็นไป<r>[e][N]เอง</r>\n'

We can see the System and Instruction prompts are successfully masked!

In [22]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA RTX A5000. Max memory = 23.673 GB.
7.916 GB of memory reserved.


In [23]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,970 | Num Epochs = 3 | Total steps = 414
O^O/ \_/ \    Batch size per device = 24 | Gradient accumulation steps = 3
\        /    Data Parallel GPUs = 1 | Total batch size (24 x 3 x 1) = 72
 "-____-"     Trainable parameters = 1,235,945,472/1,235,945,472 (100.00% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
20,4.822500,4.532349
40,2.582000,2.592261
60,2.139100,2.161409
80,1.959300,1.960382
100,1.767300,1.837112
120,1.715200,1.723563
140,1.350000,1.709559
160,1.433100,1.637149
180,1.429600,1.601025
200,1.375700,1.563413



=== Generating output at step 20 ===
Input: <klon8> จะรักน้อง
Output: <|begin_of_text|><klon8> จะรักน้องเป็นที่เคืองลวงขนินงู พานกุญชราเภอมาตยักตรอมยินชำฉันราเอ็งระทันแจ้งเสมอเสมอเจ้าเบาไม้ออกมาคลาสง่ายงามกว่าเจ้าแม่แก้ไข่นตรอื้นพลอย่าเข็งนั้นลาเหลือมราเมียที่เรียกพราหมอบก็หมายจีนหมีพรอมาไม้เจ้าประดับดีเที่ยวการะทั่งเอนุชาวโง่นมินáเวียนยินราแล้วเสริมไยกระดีจวนกินกินย่านเสาเป็นโญษาไร่เจียกระเฉยีพระยุ่งหมายหรือครันเดียเหลือเปี้ยวนี้ลั่งคิดอื้อนิจจอดวันคะนี้ที่แท้ด่านซื่อศิลาครันไม้ฉันนาบัวพลายน์หยุดธา พูนโหล่อรวมเขนีมารวยถีหราเกรงดีละมราเมียระคิดอินสานะมาศยืนมั่นปิดจึ่งไพล่นคาเขือวาไม้ป่าลีหนักตราเห็นหรรี่ดีมี่ดีเหลือเมียใหม่หวายหน้าม้าธุรงยิบตาเมียศักมราใหญ่ไกล่ดีเส้นเที่ยวตรีเนื้อคะวาเถราแท้จำป่าอินทรีระเทียมประหลาดตัวอินเทียกว่าซ้ำผู้สู้หน้าพลอดคงระหัสไร้ค่ายห้ามถูกซื่อตรองชื่อม้าแข็งราเอ๋ยหมากาเผ่าห้ามมราเนื้อเนื้อมือจูฬาฯกูปราปรีษาประหลาดใจจ้าดูวิแม่แคล้วเชิญเอ๋ยวันทองคร


Unsloth: Not an error, but LlamaModel does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



=== Generating output at step 40 ===
Input: <klon8> แต่ไม่มีกิ
Output: <|begin_of_text|><klon8> แต่ไม่มีกิ๋อต่อเนื้อความไม่แถงนาทำเร่งจอมยากรุ่งพาน้องเลือดนตรมริมภิเฝ้าแกล้งเมียกตัวสาครั้งนี่ทั้งเมืองเขาเจ้าเจ้าเจนทา[n][a][n][o][x][i][a][a][a]<r>[a][t][i][i][t][O][i][N][a][a][a][a][a][N][N][a]</r>[n][n][a][n][a]<r>[a][o][m][a][N][i][a]<r></r>[N][j][i][i][t][a][j][a][N]<r></r>[a][a][n][a][a]<r>[a]<r></r>[j][a][a][n][O][a][a][i][a][a][j][i][a][a][N][N][a][j][t][a][j][t][n]</r>[a][a][a][a][a][i][a][i][i][a][a][a][n][i][a][o]<r>[a][a]</r>[t][p][a][j][i][a][j][n][j][a][O]<r>[a][N][i][o][a][a][a][N][n][j]<r>[j]<r>[n][a][n][a][a]</r>[a][a][i]</r>[p][a][i][N]</r>[j][i][a][N][j][n][j][n][i][t]<r>[a][o][N][N][a]<r>[n]<r>[m][a][a][a]<r>[j]<r>[w][O][a][p][a][j][a][j][n][n][j][n][m][j][a][N][a][a][n][o][a][i][n][a][a][t][i][t][n][a][a][a][a][a][a]<r>[t][o][j][a][U]<r>[i]<r>[j][w][a][n][t][j][i]<r></r></r><r></r>[j]</r>[t][a]<r>[N][u][u][a][i]</r>[j][a][a][i][a][j][j][n][n]<r>[N][n][t]<r>[O][i]<r>[

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [24]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

2005.2488 seconds used for training.
33.42 minutes used for training.
Peak reserved memory = 22.666 GB.
Peak reserved memory for training = 14.75 GB.
Peak reserved memory % of max memory = 95.746 %.
Peak reserved memory for training % of max memory = 62.307 %.


In [25]:
from transformers import TextStreamer

sample_text = "<klon8> พอได้ยินเสียงระฆัง"
max_new_tokens = 1028
temperature = 1
top_p = 0.95
top_k = 64

# Prepare inputs
inputs = tokenizer(
    [sample_text],
    return_tensors="pt",
    padding=True,
    truncation=True
).to(model.device)

# Generate outputs
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        pad_token_id=tokenizer.eos_token_id,
        streamer=TextStreamer(tokenizer, skip_prompt = False),
    )

# # Decode and print
# generated_text = tokenizer.decode(
#     outputs[0],
#     skip_special_tokens=False,
#     clean_up_tokenization_spaces=True
# )

# print(f"Input: {sample_text}")
# print(f"Output: {generated_text}")
# print("=" * 80)

<|begin_of_text|><klon8> พอได้ยินเสียงระฆังป้องคำนับลงไปห้ำระเรือกล่อมะโกรา<r>[i][a][p]พี่</r>[O][a][a][a][n]สัญ<r>[a][a][t]ฎา</r>[i][a][n]อินทร์</r>[u][n]อุรา</r>[a][a][n]ทั่ง</r>[a][a][n]วรรณ</r>[a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][O][a][n]ศรี<r>[i]วี<r>[a][j]ศีรอง<r>[O][n][n][n]ร่อนุชี<r>[a][j]ลัย</r>[u][n][N]ถุน<r>[a][m][N][N][u][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][ua][j]ช<r>[a][j]ลัย</r>[a][i][n]ศีรื้อห้า<r>[a][j]อา<r>[a][n]ศีร<r>[a][p][t][o][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][O][a][n]พลั้น<r>[o][a][n]กุล<r>[a][n]<r>[a][j]ไฟ</r>[a][a][n][n][n]<r>[a][p][n][a][a][a][O][o][n][n][n]ผู้<r>[i][a][N][N][O][a][a][n][t][u][k][N][u][ua][j]ขาว<r>[O][N][n][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][a][n]ภิ<r>[O][n][n][n]หม่อม<r>[a][j]ทัย</r>[i][iaศีร้าย<r>[o][n][n][n]<r>[o][

In [36]:
import gc

# Clear deleted GPU items
for _ in range(3):
    gc.collect()
    torch.cuda.empty_cache()

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

**[NEW] Try 2x faster inference in a free Colab for Llama-3.1 8b Instruct [here](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Unsloth_Studio.ipynb)**

We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [26]:
model.vocab_size, tokenizer.vocab_size

(128320, 128000)

In [27]:
hub_model_id= f"Pongsaky/{experiment_name}"
hub_model_id

'Pongsaky/llama3.2-typhoon2-1b-full-training-phonetic'

In [28]:
model.save_pretrained(experiment_name)  # Local saving
tokenizer.save_pretrained(experiment_name)
model.push_to_hub(hub_model_id, token = HUGGINGFACE_API_KEY) # Online saving
tokenizer.push_to_hub(hub_model_id, token = HUGGINGFACE_API_KEY) # Online saving

README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Saved model to https://huggingface.co/Pongsaky/llama3.2-typhoon2-1b-full-training-phonetic


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [ ]:
import json


def get_data_from_json(file_path):
  with open(file_path, "r", encoding="utf-8") as f:
      data = json.load(f)
  return data


test_data_input = get_data_from_json("./data/test_inputs.json")

In [ ]:
import torch
from transformers import TextStreamer
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
is_adapter = True

if True:
    from unsloth import FastLanguageModel
    model_name = None
    if not is_adapter:
        model_name = "Pongsaky/llama3.2-typhoon2-1b-full-training-phonetic"
    else:
        model_name = "scb10x/llama3.2-typhoon2-1b",
        
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length = 1024,
        load_in_4bit = False,
    )

    if not is_adapter:
        model = base_model
    else:
        # Added new token for adapter merging with base model
        from unsloth import add_new_tokens
        add_new_tokens(base_model, tokenizer, phonetic_token)

        peft_model_id = "Pongsaky/llama3.2-typhoon2-1b-lora-unfreeze-embedding-phonetic"
        model = PeftModel.from_pretrained(base_model, peft_model_id)

sample_text = "<klon8> พอได้ยินเสียงระฆัง"
max_new_tokens = 512
temperature = 1
top_p = 0.95
top_k = 64

rand_idx = int(torch.rand(1) * len(test_data_input))
sample_text = test_data_input[rand_idx]

# Prepare inputs
inputs = tokenizer(
    [sample_text],
    return_tensors="pt",
    padding=True,
    truncation=True
).to(model.device)

# Generate outputs
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        pad_token_id=tokenizer.eos_token_id,
        streamer=TextStreamer(tokenizer, skip_prompt = False),
    )

==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.673 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
<|begin_of_text|><klon8> อยู่ที่นี่เห็นหึงคุณท่าน<r>[x][k]เอ๋ย</r>	พ่อจะแก้<r>[x][w]แก้</r>ดีจริงเห็นมิ<r>[a]หน้า</r>
เสียแรงเป็นผู้ใหญ่สอนศีล<r>[a][n]ธรรม์</r>	ทำกลับ<r>[a][n]บ่อ</r>ทำหักหลังเป็นหนัก<r>[a]หนา</r>
เพราะผู้ดีคิดทำดีให้<r>[a]หา</r>	ไม่สม<r>[a][t]ภาณ์</r>เหมือนท่านให้ศีล<r>[a][j]สั่ง</r>
มิได้บอกความผิดดังท่านสอน<r>[a][j]ไว้</r>	ฉวยทำ<r>[a][j]ไว้</r>อยู่เป็นบ่าว<r>[ua][w]เฒ่า</r>
ผู้ดีมีศักดินาทั้ง<r>[i]ที</r>	พอเดิน<r>[i]ถึง</r>มีธุระต้องตัก<r>[ua][n]เตือน</r>
มิควรขืนคิดทำลายเสีย<r>[ua][n]เกา</r>	ด้วยท่าน<r>[ua][n]

In [7]:
commit_message = "epoch-4"
hub_model_id = "Pongsaky/llama3.2-typhoon2-1b-lora-unfreeze-embedding-phonetic"

model.push_to_hub(hub_model_id, token = HUGGINGFACE_API_KEY, commit_message="add model " + commit_message) # Online saving
tokenizer.push_to_hub(hub_model_id, token = HUGGINGFACE_API_KEY, commit_message="add tokenizer " + commit_message) # Online saving

/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:250: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Saved model to https://huggingface.co/Pongsaky/llama3.2-typhoon2-1b-lora-unfreeze-embedding-phonetic


No files have been modified since last commit. Skipping to prevent empty commit.


In [65]:
model.vocab_size, tokenizer.vocab_size

(128320, 128000)

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
